<a href="https://colab.research.google.com/github/koderlad/assignment-M508D/blob/main/M508D_GH1050620.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assignment started on 30 April 2026.

I am using the dataset from hugging face [https://huggingface.co/datasets/ihmz/enhanced_hiring_data_completeV2]

## **Installing Required Dependencies**
This section is only used to install required dependency.

In [22]:
# !pip install spacy

##**Loading Dependencies**

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import spacy
import re ##Regular Expression to parse data

## **Loading Dataset**

In [24]:
df = pd.read_json('hf://datasets/ihmz/enhanced_hiring_data_completeV2/hiring_chat.json')

### **REGEX PATTERNS**

In [25]:
FIELD_PATTERN = re.compile(
    r"Role:\s*(?P<role>.*?)\n"
    r"Job_Description:\s*(?P<jd>.*?)\n"
    r"Resume_Masked:\s*(?P<resume>.*?)\n"
    r"Transcript_Masked:\s*(?P<transcript>.*)",
    re.DOTALL,
)

In [26]:
SPEAKER_SPLIT = re.compile(r"(Interviewer|\[NAME\]):")

In [27]:
RESUME_SECTION_PATTERN = re.compile(r"\n([A-Z][A-Za-z ]+):\s*\n", re.MULTILINE)

## **PARSING FUNCTIONS**

In [46]:
def parse_user_content(content: str):
  #First lets split the high-level topics from the conversation.
  m = FIELD_PATTERN.search(content)
  if not m:
    return None
  return {
      'role': m.group('role').strip(),
      "job_description": m.group('jd').strip(),
      "resume": m.group('resume').strip(),
      "transcript": m.group('transcript').strip(),
  }

In [47]:
def parse_transcript_turns(transcript: str):
  parts = SPEAKER_SPLIT.split(transcript)
  turns = []
  for i in range(1, len(parts) - 1, 2):
    speaker = parts[i]
    content = parts[i + 1].strip()
    if content:
      turns.append({'speaker': speaker, "content": content})
  return turns

In [48]:
def parse_resume_sections(resume: str):
  matches = list(RESUME_SECTION_PATTERN.finditer(resume))
  sections = {}
  for i, mh in enumerate(matches):
    name = mh.group(1).strip().lower().replace(" ", "_")
    start = mh.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(resume)
    sections[name] = resume[start:end].strip()
  return sections


In [49]:
def normalise_label(raw_label: str):
    return raw_label.strip().lower().rstrip(".!?\"'")

In [50]:
def process_row(conversations:list):
  if not conversations or len(conversations) < 2:
    return None

  user_msg, assistant_msg = conversations[0], conversations[1]
  user_content = user_msg.get("content", "")
  assistant_content = assistant_msg.get("content", "")

  # print("User Content: \n", user_content)
  # print("Assistant Content: ", assistant_content)

  fields = parse_user_content(user_content)
  if not fields:
    return None

  transcript_turns = parse_transcript_turns(fields["transcript"])
  resume_sections = parse_resume_sections(fields["resume"])

  candidate_turns = [t for t in transcript_turns if t["speaker"] == "[NAME]"]
  interviewer_turns = [t for t in transcript_turns if t["speaker"] == "Interviewer"]

  return {
      "role": fields["role"],
      "job_description": fields["job_description"],
      "resume": fields["resume"],
      "transcript": fields["transcript"],
      "label": normalise_label(assistant_content),
      "label_raw": assistant_content,
      # Derived fields
      "candidate_text": " ".join(t["content"] for t in candidate_turns),
      "interviewer_text": " ".join(t["content"] for t in interviewer_turns),
      "n_turns": len(transcript_turns),
      "n_candidate_turns": len(candidate_turns),
      "resume_sections": resume_sections,
      "transcript_turns": transcript_turns,
  }

### Main Parsing

In [33]:
parsed = []
failed = 0

In [34]:
for row in df['conversations']:
  out = process_row(row)
  if out is None:
    failed += 1
  else:
    parsed.append(out)

In [35]:
df_parsed = pd.DataFrame(parsed)

In [36]:
df_parsed

,role,job_description,resume,transcript,label,label_raw,candidate_text,interviewer_text,n_turns,n_candidate_turns,resume_sections,transcript_turns
0,E-Commerce Specialist,Be part of a passionate team at the forefront ...,Here's a professional resume for [NAME]:\n\n[N...,"Interviewer: Good morning, Jason. It's great t...",reject,reject,Good morning. Thank you for having me. It's a ...,"Good morning, Jason. It's great to meet you. W...",15,7,{'contact_information': '* Email: [jasonjones@...,"[{'speaker': 'Interviewer', 'content': 'Good m..."
1,Game Developer,Help us build the next-generation products as ...,Here's a professional resume for [NAME]:\n\n[N...,Interview Scene\n\nA conference room with a ta...,select,select,"Uh, good morning. Thank you for having me. I h...","Good morning, Ann. Thank you for coming in tod...",13,6,{'contact_information': '* Email: [ann.marshal...,"[{'speaker': 'Interviewer', 'content': 'Good m..."
2,Human Resources Specialist,We need a Human Resources Specialist to enhanc...,Here's a professional resume for [NAME]:\n\n[N...,Interview Setting: A conference room in a medi...,reject,reject,"Good morning. Uh, no, thank you. I'm good. Abs...","Good morning, Patrick. Thank you for coming in...",17,8,{'contact_information': '* Phone: (555) 123-45...,"[{'speaker': 'Interviewer', 'content': 'Good m..."
3,E-Commerce Specialist,Be part of a passionate team at the forefront ...,Here's a professional resume for [NAME]:\n\n[N...,Here's a simulated professional interview for ...,select,select,"Good morning. Thank you for having me. Um, I h...","Good morning, Patricia. It's nice to meet you....",15,7,{'contact_information': '* Email: [patriciagra...,"[{'speaker': 'Interviewer', 'content': 'Good m..."
4,E-Commerce Specialist,We are looking for an experienced E-commerce S...,Here's a professional resume for [NAME]:\n\n[N...,Here's the simulated interview:\n\nInterviewer...,reject,reject,"Good morning. Um, thank you for having me. I h...","Good morning, Amanda. Thank you for coming in ...",15,7,{'contact_information': '* Email: [amanda.gros...,"[{'speaker': 'Interviewer', 'content': 'Good m..."
...,...,...,...,...,...,...,...,...,...,...,...,...
10169,Product Manager,Here is a comprehensive job description for a ...,Here's a sample resume for [NAME]:\n\n**[NAME]...,Here's a simulated interview for a Product Man...,reject,reject,"** Sure. I have a degree in marketing, and I'v...","** Hi Diana, thanks for coming in today. Can y...",15,7,{},"[{'speaker': 'Interviewer', 'content': '** Hi ..."
10170,Ui Engineer,Here is a sample job description for a UI Engi...,Here's a sample resume for [NAME]:\n\n**[NAME]...,"**Interviewer:** Hi Grace, thank you for comin...",reject,reject,"** Yeah, sure. I have a degree in Computer Sci...","** Hi Grace, thank you for coming in today. Ca...",18,9,{},"[{'speaker': 'Interviewer', 'content': '** Hi ..."
10171,Ui Engineer,Here is a job description for a UI Engineer ro...,Here's a sample resume for [NAME]:\n\n**[NAME]...,Here's a simulated interview for a UI Engineer...,select,select,** ** Thank you for having me. I have a degree...,"** Hi Hank, thank you for coming in today. Can...",18,9,{},"[{'speaker': '[NAME]', 'content': '**'}, {'spe..."
10172,Data Engineer,Here is a comprehensive job description for a ...,Here's a sample resume for [NAME]:\n\n**[NAME]...,Here's a simulated interview for a Data Engine...,reject,reject,"** Hi! Yeah, sure. I have a degree in computer...","** Hi Diana, thank you for coming in today. Ca...",14,7,{},"[{'speaker': 'Interviewer', 'content': '** Hi ..."


In [37]:
df_parsed['transcript'][0]

"Interviewer: Good morning, Jason. It's great to meet you. Welcome to the interview for the E-commerce Specialist role at our company.\n\n[NAME]: Good morning. Thank you for having me. It's a pleasure to be here.\n\nInterviewer: Before we begin, I want to let you know that this interview will cover various aspects of the role, including customer service, product listing, and other relevant topics. Can you start by telling me a little bit about your background and how you think your skills and experience make you a strong fit for this position?\n\n[NAME]: Uh, sure. I have about 3 years of experience in e-commerce, working with online marketplaces like Amazon and eBay. My most recent role was as a customer service representative for an online retailer, where I handled customer inquiries, resolved issues, and provided product recommendations. I've also worked on product listing and optimization, which I think is a key part of this role. I'm confident in my ability to understand customer n

In [38]:
df_parsed['interviewer_text'][0]

"Good morning, Jason. It's great to meet you. Welcome to the interview for the E-commerce Specialist role at our company. Before we begin, I want to let you know that this interview will cover various aspects of the role, including customer service, product listing, and other relevant topics. Can you start by telling me a little bit about your background and how you think your skills and experience make you a strong fit for this position? Great. Let's dive into customer service. Can you give me an example of a particularly challenging customer issue you had to resolve, and how you handled it? That's an excellent example. Now, let's talk about product listing. Can you walk me through your process for creating a compelling and accurate product listing? Great. How do you stay up-to-date with changes in e-commerce trends and best practices? Excellent. Finally, can you tell me why you're interested in this role and our company? Thank you, Jason, for sharing your insights and experiences wit

In [39]:
df_parsed['candidate_text'][0]

"Good morning. Thank you for having me. It's a pleasure to be here. Uh, sure. I have about 3 years of experience in e-commerce, working with online marketplaces like Amazon and eBay. My most recent role was as a customer service representative for an online retailer, where I handled customer inquiries, resolved issues, and provided product recommendations. I've also worked on product listing and optimization, which I think is a key part of this role. I'm confident in my ability to understand customer needs and provide excellent service. Ah, yeah. One time, a customer was upset because we didn't have their preferred shipping option available. They were really frustrated, and I had to listen to them vent for a while. I empathized with their situation, apologized for the inconvenience, and offered a solution – I upgraded their shipping to the next available option, which they accepted. I also provided a discount code for their next purchase, which seemed to go a long way in turning their 

In [40]:
print("Label distribution:")
print(df_parsed["label"].value_counts(dropna=False))
print("\nProportions:")
print(df_parsed["label"].value_counts(normalize=True))

Label distribution:
label
reject    5114
select    5060
Name: count, dtype: int64

Proportions:
label
reject    0.502654
select    0.497346
Name: proportion, dtype: float64


In [41]:
print("\nRoles: ")
print(df_parsed["role"].value_counts())


Roles: 
role
Data Scientist                  825
Software Engineer               787
Product Manager                 761
Data Engineer                   754
Data Analyst                    512
Ui Engineer                     507
E-Commerce Specialist           268
Devops Engineer                 266
Machine Learning Engineer       265
Human Resources Specialist      262
Digital Marketing Specialist    260
Robotics Engineer               257
Cloud Architect                 254
Qa Engineer                     251
Blockchain Developer            251
Mobile App Developer            247
Full Stack Developer            246
Database Administrator          243
Cloud Engineer                  240
Game Developer                  239
Content Writer                  238
Ar/Vr Developer                 237
Cybersecurity Analyst           234
Ux Designer                     227
Ui Designer                     226
Business Analyst                226
Ai Researcher                   225
Data Architect

In [42]:
df_parsed["resume_chars"] = df_parsed["resume"].str.len()
df_parsed["transcript_chars"] = df_parsed["transcript"].str.len()
df_parsed["candidate_chars"] = df_parsed["candidate_text"].str.len()
print("\nLength stats:")
print(df_parsed[["resume_chars", "transcript_chars", "candidate_chars", "n_candidate_turns"]].describe())


Length stats:
       resume_chars  transcript_chars  candidate_chars  n_candidate_turns
count  10174.000000      10174.000000     10174.000000       10174.000000
mean    2877.275703       4261.975624      2761.005308           7.177118
std      496.527009        793.280405      1017.778383           2.297940
min       29.000000        298.000000         0.000000           0.000000
25%     2692.000000       3923.000000      2456.000000           7.000000
50%     2909.000000       4315.000000      2790.000000           7.000000
75%     3132.000000       4708.000000      3181.750000           8.000000
max     4681.000000       7671.000000      7173.000000          19.000000


In [43]:
print("\n--- Sample rejects ---")
for txt in df_parsed[df_parsed["label"] == "reject"]["candidate_text"].head(2):
    print(txt[:300], "\n---")

print("\n--- Sample selects ---")
for txt in df_parsed[df_parsed["label"] == "select"]["candidate_text"].head(2):
    print(txt[:300], "\n---")


--- Sample rejects ---
Good morning. Thank you for having me. It's a pleasure to be here. Uh, sure. I have about 3 years of experience in e-commerce, working with online marketplaces like Amazon and eBay. My most recent role was as a customer service representative for an online retailer, where I handled customer inquirie 
---
Good morning. Uh, no, thank you. I'm good. Absolutely. I have about 5 years of experience in HR, most recently as an HR Generalist for a non-profit organization. I've worked on everything from recruitment and benefits administration to training and employee relations. I'm excited about this role bec 
---

--- Sample selects ---
Uh, good morning. Thank you for having me. I have a background in computer science and game development. I'm excited about this role because I love creating immersive experiences for players. I've worked on several projects in my free time, including a 3D modeling course I completed recently. (pause 
---
Good morning. Thank you for having

In [44]:
df_parsed.head(2)

,role,job_description,resume,transcript,label,label_raw,candidate_text,interviewer_text,n_turns,n_candidate_turns,resume_sections,transcript_turns,resume_chars,transcript_chars,candidate_chars
0,E-Commerce Specialist,Be part of a passionate team at the forefront ...,Here's a professional resume for [NAME]:\n\n[N...,"Interviewer: Good morning, Jason. It's great t...",reject,reject,Good morning. Thank you for having me. It's a ...,"Good morning, Jason. It's great to meet you. W...",15,7,{'contact_information': '* Email: [jasonjones@...,"[{'speaker': 'Interviewer', 'content': 'Good m...",2603,3743,2431
1,Game Developer,Help us build the next-generation products as ...,Here's a professional resume for [NAME]:\n\n[N...,Interview Scene\n\nA conference room with a ta...,select,select,"Uh, good morning. Thank you for having me. I h...","Good morning, Ann. Thank you for coming in tod...",13,6,{'contact_information': '* Email: [ann.marshal...,"[{'speaker': 'Interviewer', 'content': 'Good m...",417,3830,2313
